# p003 — Predicciones de 2 modelos

Cada modelo usa su dataset y RDS. Los meses, semillas y columnas excluidas son compartidos. Se promedian todas las probabilidades por cliente antes de ordenar y generar el corte.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import rdata
from IPython.display import display
import os


## Directorios

In [2]:
carpeta_entregas = Path.cwd()
data_folder = carpeta_entregas / "dataset"
carpeta_parametros = carpeta_entregas / "parametros"
carpeta_salida = carpeta_entregas / "entregas"
archivo_logging = carpeta_entregas / "logging_python.csv"


In [3]:
os.listdir(data_folder)


['versiones_dataset_2.ipynb',
 'c_01_tern_2_lf_k6_p4.csv',
 'c_01_tern_2_lf_O2_k3_p6.csv',
 'competencia_01_ternaria_lf_k6_perdida4.csv',
 'c_01_tern_2_lags_O2.csv',
 'versiones_dataset_1.ipynb',
 'c_01_tern_2_lags.csv',
 'competencia_01_ternaria.csv',
 'RANKS_LAGS_O2.csv',
 'competencia_01_crudo.csv',
 'competencia_01_ternaria_lags.csv',
 'RANKS_LF_O2_K6_P4.csv',
 'c_01_tern_2.csv']

## Parámetros

Completar los dos datasets y RDS en el mismo orden: modelo 1, modelo 2.

In [4]:
dataset_files = [
    "c_01_tern_2_lf_k6_p4.csv",  # dataset del modelo 1, con .csv
    "c_01_tern_2_lf_O2_k3_p6.csv",  # dataset del modelo 2, con .csv
]
ids_rds = [
    40,  # id del RDS del modelo 1
    40,  # id del RDS del modelo 2
]

train_final = [202103, 202104, 202105, 202106]
future = [202108]
# SEEDS = [
#     230047, 230059, 230063, 230077, 230081,
#     230089, 230101, 230107, 230117, 230123,
# ]
SEEDS = [
    230047, 230059, 230063, 230077, 230081,
    230089, 230101, 230107, 230117, 230123,
    230137, 230143, 230149, 230189, 230203,
    230213, 230221, 230227, 230233, 230239,
]


In [5]:
columnas_excluir = [
    "ternaria",
    "ternaria_lag1",
    "clase01",
    "fold",
    "azar",
    "training",
    "cprestamos_personales",
    "mprestamos_personales",
    "cprestamos_personales_lag1",
    "mprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "mprestamos_personales_delta1",
    "cprestamos_personales_lag_pond",
    "mprestamos_personales_lag_pond",
    "cprestamos_personales_delta_lag_pond",
    "mprestamos_personales_delta_lag_pond"
]


## Lectura de parámetros y preparación de datos

In [6]:
def convertir_r(valor):
    if isinstance(valor, dict):
        return {k: convertir_r(v) for k, v in valor.items()}
    if isinstance(valor, np.ndarray):
        if valor.size == 1:
            return convertir_r(valor.item())
        return [convertir_r(v) for v in valor.tolist()]
    if isinstance(valor, (list, tuple)):
        return [convertir_r(v) for v in valor]
    if isinstance(valor, np.generic):
        return convertir_r(valor.item())
    # R serializa muchos parámetros enteros como double (31.0).
    # LightGBM Python exige int para esos valores; se conserva su valor.
    if isinstance(valor, float) and np.isfinite(valor) and valor.is_integer():
        return int(valor)
    return valor

def data_matrix(df):
    matriz = df.copy()
    for nombre in matriz.columns:
        serie = matriz[nombre]
        if isinstance(serie.dtype, pd.CategoricalDtype):
            codigos = serie.cat.codes
            matriz[nombre] = (codigos + 1).where(codigos >= 0, np.nan)
        elif pd.api.types.is_object_dtype(serie.dtype) or pd.api.types.is_string_dtype(serie.dtype):
            niveles = sorted(serie.dropna().unique())
            codigos = pd.Categorical(serie, categories=niveles).codes
            matriz[nombre] = np.where(codigos >= 0, codigos + 1, np.nan)
    return matriz.to_numpy(dtype=np.float64)


## Entrenamiento y predicciones

Los modelos se procesan uno después del otro. Las predicciones de cada semilla quedan en memoria.

In [7]:
if (len(dataset_files) != 2 or len(ids_rds) != 2
        or any(not archivo for archivo in dataset_files)
        or any(id_rds is None for id_rds in ids_rds)):
    raise ValueError("Completar los dos datasets y sus dos ids_rds, en el mismo orden.")

if len(future) != 1 or not SEEDS or len(SEEDS) != len(set(SEEDS)):
    raise ValueError("Se requiere un mes a predecir y una lista de semillas sin repetidos.")

predicciones_por_modelo = []
cantidad_registros_totales = []
cantidad_registros_entrenamiento = []

for numero_modelo, (dataset_file, id_rds) in enumerate(zip(dataset_files, ids_rds), 1):
    dataset = pd.read_csv(data_folder / dataset_file, low_memory=False)
    dataset["ternaria"] = dataset["ternaria"].fillna("")
    dataset = dataset.sort_values(
        ["foto_mes", "ternaria", "numero_de_cliente"],
        kind="stable", na_position="first",
    ).reset_index(drop=True)

    parametros = convertir_r(rdata.read_rds(carpeta_parametros / f"{id_rds}.rds"))
    params_base = (
        parametros
        if "num_iterations" in parametros
        else next(iter(parametros.values()))
    )

    dataset["clase01"] = dataset["ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int32)
    campos_buenos = [c for c in dataset.columns if c not in columnas_excluir]
    dataset_train = dataset.loc[dataset["foto_mes"].isin(train_final)].copy()
    dfuture = dataset.loc[dataset["foto_mes"].isin(future)].copy()

    if dataset_train.empty or dfuture.empty:
        raise ValueError(f"Modelo {numero_modelo}: no hay filas para los meses elegidos.")

    tb_predicciones = dfuture[["numero_de_cliente"]].set_index("numero_de_cliente")
    if not tb_predicciones.index.is_unique or tb_predicciones.index.hasnans:
        raise ValueError(f"Modelo {numero_modelo}: revisar los identificadores de clientes en future.")

    if predicciones_por_modelo and not tb_predicciones.index.sort_values().equals(
        predicciones_por_modelo[0].index.sort_values()
    ):
        raise ValueError("Los dos datasets tienen distintos clientes para el mes a predecir.")

    cantidad_registros_totales.append(len(dataset))
    cantidad_registros_entrenamiento.append(len(dataset_train))

    display(dataset_train.groupby(["foto_mes", "ternaria"], dropna=False).size().reset_index(name="N"))

    X_train = data_matrix(dataset_train[campos_buenos])
    X_future = data_matrix(dfuture[campos_buenos])

    for seed in SEEDS:
        params_semilla = params_base.copy()
        params_semilla["seed"] = seed
        num_boost_round = int(params_semilla.pop("num_iterations"))

        dtrain_final = lgb.Dataset(
            data=X_train,
            label=dataset_train["clase01"].to_numpy(),
            feature_name=campos_buenos,
            categorical_feature=[],
        )
        modelo_final = lgb.train(
            params=params_semilla,
            train_set=dtrain_final,
            num_boost_round=num_boost_round,
        )

        tb_predicciones[f"prob_{seed}"] = modelo_final.predict(X_future)
        print(f"Modelo={numero_modelo} Semilla={seed}: predicciones listas")
        del dtrain_final, modelo_final

    predicciones_por_modelo.append(tb_predicciones)
    del dataset, dataset_train, dfuture, X_train, X_future


/tmp/ipykernel_376095/3997393612.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset["clase01"] = dataset["ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int32)


,foto_mes,ternaria,N
0,202103,BAJA+1,1019
1,202103,BAJA+2,960
2,202103,continua,160921
3,202104,BAJA+1,964
4,202104,BAJA+2,1139
5,202104,continua,161181
6,202105,BAJA+1,1143
7,202105,BAJA+2,870
8,202105,continua,161755
9,202106,BAJA+1,874


Modelo=1 Semilla=230047: predicciones listas
Modelo=1 Semilla=230059: predicciones listas
Modelo=1 Semilla=230063: predicciones listas
Modelo=1 Semilla=230077: predicciones listas
Modelo=1 Semilla=230081: predicciones listas
Modelo=1 Semilla=230089: predicciones listas
Modelo=1 Semilla=230101: predicciones listas
Modelo=1 Semilla=230107: predicciones listas
Modelo=1 Semilla=230117: predicciones listas
Modelo=1 Semilla=230123: predicciones listas
Modelo=1 Semilla=230137: predicciones listas
Modelo=1 Semilla=230143: predicciones listas
Modelo=1 Semilla=230149: predicciones listas
Modelo=1 Semilla=230189: predicciones listas
Modelo=1 Semilla=230203: predicciones listas
Modelo=1 Semilla=230213: predicciones listas
Modelo=1 Semilla=230221: predicciones listas
Modelo=1 Semilla=230227: predicciones listas
Modelo=1 Semilla=230233: predicciones listas
Modelo=1 Semilla=230239: predicciones listas


/tmp/ipykernel_376095/3997393612.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset["clase01"] = dataset["ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int32)


,foto_mes,ternaria,N
0,202103,BAJA+1,1019
1,202103,BAJA+2,960
2,202103,continua,160921
3,202104,BAJA+1,964
4,202104,BAJA+2,1139
5,202104,continua,161181
6,202105,BAJA+1,1143
7,202105,BAJA+2,870
8,202105,continua,161755
9,202106,BAJA+1,874


Modelo=2 Semilla=230047: predicciones listas
Modelo=2 Semilla=230059: predicciones listas
Modelo=2 Semilla=230063: predicciones listas
Modelo=2 Semilla=230077: predicciones listas
Modelo=2 Semilla=230081: predicciones listas
Modelo=2 Semilla=230089: predicciones listas
Modelo=2 Semilla=230101: predicciones listas
Modelo=2 Semilla=230107: predicciones listas
Modelo=2 Semilla=230117: predicciones listas
Modelo=2 Semilla=230123: predicciones listas
Modelo=2 Semilla=230137: predicciones listas
Modelo=2 Semilla=230143: predicciones listas
Modelo=2 Semilla=230149: predicciones listas
Modelo=2 Semilla=230189: predicciones listas
Modelo=2 Semilla=230203: predicciones listas
Modelo=2 Semilla=230213: predicciones listas
Modelo=2 Semilla=230221: predicciones listas
Modelo=2 Semilla=230227: predicciones listas
Modelo=2 Semilla=230233: predicciones listas
Modelo=2 Semilla=230239: predicciones listas


## Combinación por cliente

In [9]:
tb_prediccion = (
    pd.concat(predicciones_por_modelo, axis=1)
    .mean(axis=1)
    .rename("prob")
    .reset_index()
    .sort_values("prob", ascending=False, kind="stable")
    .reset_index(drop=True)
)


## Generación del corte

Cambiar `envios` y ejecutar esta celda para generar otro corte sin entrenar de nuevo. Registrar cada generación en la celda de documentación antes de generar la siguiente, para que avance `id_prueba`.

In [18]:
envios = 12500  

if envios != int(envios) or not 1 <= envios <= len(tb_prediccion):
    raise ValueError("Revisar envios: debe ser un entero entre 1 y la cantidad de clientes.")
envios = int(envios)

if archivo_logging.exists():
    historico = pd.read_csv(archivo_logging)
    id_prueba = int(historico["id_prueba"].max()) + 1
else:
    id_prueba = 1

carpeta_salida.mkdir(parents=True, exist_ok=True)
archivo = f"p003_2models_{id_prueba}.csv"
tb_prediccion.head(envios)[["numero_de_cliente"]].to_csv(
    carpeta_salida / archivo,
    header=False,
    index=False,
)
archivos_salida = [archivo]
print(f"Archivo={archivo} Envios={envios}")


Archivo=p003_2models_47.csv Envios=12500


In [ ]:
pepe

## Documentación

Datasets, RDS y cantidades de registros por modelo se guardan separados por comas, en el mismo orden que los parámetros. La cantidad de registros de predicción corresponde a los clientes de la combinación final. Los datasets se registran sin `.csv`.

In [19]:
nota = "primer ensamble, rds40, 2 datasets, subimos envios a 12.5"
resultado_competencia_mean = "107.4425"  # score de Kaggle o referencia a un problema con la entrega
resultado_competencia_std = "NA"
ref_submit = "e47_p"


In [20]:
registro_prueba = pd.DataFrame([{
    "id_prueba": id_prueba,
    "ref_submit": ref_submit,
    "notebook": "p003_2models.ipynb",
    "dataset": ",".join(Path(archivo).stem for archivo in dataset_files),
    "seeds": ",".join(map(str, SEEDS)),
    "cantidad_registros_totales": ",".join(map(str, cantidad_registros_totales)),
    "cantidad_registros_entrenamiento": ",".join(map(str, cantidad_registros_entrenamiento)),
    "cantidad_registros_prediccion": len(tb_prediccion),
    "meses_entrenamiento": ",".join(map(str, train_final)),
    "meses_prediccion": ",".join(map(str, future)),
    "columnas_excluidas": ",".join(columnas_excluir),
    "bajas":"1+2",                                       #baja+1 y baja+2 o solo baja+2
    "id_rds": ",".join(map(str, ids_rds)),
    "envios": envios,
    "archivos": ",".join(archivos_salida),
    "nota": nota,
    "resultado_competencia_mean": resultado_competencia_mean,
    "resultado_competencia_std": resultado_competencia_std
}])

if archivo_logging.exists():
    historico = pd.read_csv(
        archivo_logging,
        dtype={
            "ref_submit": str,
            "id_rds": str,
            "cantidad_registros_totales": str,
            "cantidad_registros_entrenamiento": str,
            "seeds": str,
            "meses_entrenamiento": str,
            "meses_prediccion": str,
            "nota": str,
            "resultado_competencia_mean": str,
            "resultado_competencia_std": str,
        },
    )
    fila = historico["id_prueba"].eq(id_prueba)

    if fila.any():
        historico.loc[fila, registro_prueba.columns] = registro_prueba.to_numpy()
        historico.to_csv(archivo_logging, index=False)
    else:
        registro_prueba.to_csv(archivo_logging, mode="a", header=False, index=False)
else:
    registro_prueba.to_csv(archivo_logging, index=False)

# display(registro_prueba)


In [23]:
display(pd.read_csv(archivo_logging)[['id_prueba','envios','resultado_competencia_mean','resultado_competencia_std', 'ref_submit','dataset','meses_entrenamiento','id_rds','nota']].tail(30))

,id_prueba,envios,resultado_competencia_mean,resultado_competencia_std,ref_submit,dataset,meses_entrenamiento,id_rds,nota
17,18,11500,94.7622,2.3842,e18_p,c_01_tern_2_lags.csv,"202103,202104,202105,202106",56,"rds 56, mismo anterior, bajamos a 11.5k envios"
18,19,12500,95.6395,1.7826,e19_p,c_01_tern_2_lags.csv,"202103,202104,202105,202106",56,"rds 56, mismo anterior, subimos a 12.5k envios"
19,20,11500,106.5983,1.7854,e20_p,c_01_tern_2_lf_O2_k3_p6.csv,"202103,202104,202105,202106",58,"rds 58, 12.5k envios"
20,21,11000,106.0153,1.9646,e21_p,c_01_tern_2_lf_O2_k3_p6.csv,"202103,202104,202105,202106",58,"rds 58, mismo anterior, bajamos a 11k envios"
21,22,12000,106.4718,1.9646,e22_p,c_01_tern_2_lf_O2_k3_p6.csv,"202103,202104,202105,202106",58,"rds 58, mismo anterior, submis a 12k envios"
22,23,12000,93.8905,3.0705,e23_p,c_01_tern_2_lags_O2.csv,"202103,202104,202105,202106",60,"rds 60, 12k envios, hp para lags O2 normales"
23,24,11000,91.3907,2.0535,e24_p,c_01_tern_2_lags_O2.csv,"202103,202104,202105,202106",60,"rds 60, mismo anterior, bajamos a 11k envios"
24,25,13000,95.2380,3.3412,e25_p,c_01_tern_2_lags_O2.csv,"202103,202104,202105,202106",60,"rds 60, mismo anterior, subimos a 13k envios"
25,26,14000,95.8815,2.5154,e26_p,c_01_tern_2_lags_O2.csv,"202103,202104,202105,202106",60,"rds 60, mismo anterior, subimos a 14k envios"
26,27,11000,106.7550,2.8206,e27_p,c_01_tern_2_lf_O2_k3_p6.csv,"202103,202104,202105,202106",40,"rds 40, ahora con lags frac de orden 2, 11k en..."
